# 📗 LangChain RAG 와 Text-to-SQL — 검색과 조회를 도구로

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

**RAG 를 만든 앞선 단원**에서 우리는 검색을 **손으로** 만들었습니다. 문서를 읽어 잘라 두고, 임베딩 모델로 벡터를 만들고, 벡터DB 에 직접 넣고 꺼냈습니다. 이번 시간에는 **똑같은 일을 LangChain 부품으로** 조립합니다. 부품으로 만들면 세 가지가 달라집니다 — 부품을 **갈아 끼울 수 있고**, 파이프 `|` 로 **체인**이 되고, 무엇보다 **도구로 감싸 에이전트에 붙일 수 있습니다.**

이어서 **Text-to-SQL** 을 만듭니다. 자연어 질문을 모델이 SQL 로 옮기고, 우리가 그 SQL 을 실행해 답하는 구조입니다. 남이 만든 SQL 을 실행하는 일이니 **무엇을 못 하게 막을지**도 함께 정합니다.

## ⏪ 복습 — 지난 시간: 구조화된 출력·도구·에이전트

- **구조화된 출력**: `model.with_structured_output(스키마)` 로 답을 **정해진 모양**으로 받았습니다.
- **도구(`@tool`)**: 파이썬 함수에 `@tool` 을 붙이면 **docstring 과 타입힌트가 모델에게 가는 명세**가 됐습니다.
- **에이전트(`create_agent`)**: 모델이 **도구를 쓸지 말지, 어떤 인자로 부를지** 스스로 정했습니다.
- **메시지 궤적**: `사람 → AI(도구 호출) → 도구 결과 → AI(최종 답)` 이 `messages` 에 그대로 남았습니다.

오늘은 여기에 **검색(RAG)** 과 **데이터베이스 조회** 를 도구로 얹습니다.

### 📚 공식 문서 — 오늘 배우는 것들

| 오늘 다루는 것 | 공식 문서 |
|---|---|
| 검색 부품 전체(스플리터·임베딩·벡터스토어·리트리버) | [Retrieval](https://docs.langchain.com/oss/python/langchain/retrieval) |
| RAG 구조 | [RAG](https://docs.langchain.com/oss/python/langchain/rag) · [Knowledge base](https://docs.langchain.com/oss/python/langchain/knowledge-base) |
| Chroma 벡터스토어 | [Chroma 연동](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma) |
| HuggingFace 임베딩 | [HuggingFace 임베딩 연동](https://docs.langchain.com/oss/python/integrations/text_embedding/huggingfacehub) |
| 텍스트 스플리터 인자 | [langchain-text-splitters 레퍼런스](https://reference.langchain.com/python/langchain-text-splitters/) |
| 도구로 감싸기 | [Tools](https://docs.langchain.com/oss/python/langchain/tools) · [`@tool` 레퍼런스](https://reference.langchain.com/python/langchain-core/tools/) |
| 에이전트 | [Agents](https://docs.langchain.com/oss/python/langchain/agents) |
| 구조화된 출력으로 받기 | [Models](https://docs.langchain.com/oss/python/langchain/models) |

> 두 사이트의 역할이 다릅니다. `docs.langchain.com` 은 **개념과 사용법을 설명하는 안내서**이고, `reference.langchain.com` 은 **클래스와 인자를 그대로 나열한 사전**입니다. 처음 배울 때는 안내서를, 인자 이름이 헷갈릴 때는 사전을 보세요.

**오늘의 목표**

- [ ] **부품으로 조립** — `Document`·`TextSplitter`·`Embeddings`·`VectorStore`·`Retriever` 로 색인을 만든다.
- [ ] **RAG 체인** — 검색과 생성을 LCEL 로 잇고, 답과 함께 **출처**를 돌려받는다.
- [ ] **체인의 성질** — 파이프로 고정된 경로는 인사말에도 검색이 돈다는 것을 눈으로 확인한다.
- [ ] **Text-to-SQL** — 표 구조를 알려 주고 모델이 만든 SQL 을 **안전하게** 실행한다.
- [ ] **두 겹 가드** — 문자열 검사와 읽기 전용 연결이 각각 무엇을 막는지 실제로 확인한다.
- [ ] **구조화된 출력으로 SQL 받기** — SQL 과 함께 근거·사용한 표까지 받아 로그로 남긴다.

아래 준비 셀을 먼저 실행하세요(앞 시간과 같은 `.env` 의 `OPENAI_API_KEY` 를 씁니다).

In [ ]:
# [제공 코드] OpenAI 키 준비 — 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다 — 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 — 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 이번 시간 공통 부품 — 앞 시간에 배운 모델
from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

---
# 1. RAG 를 LangChain 부품으로 — Document·Splitter·Embeddings·VectorStore·Retriever

## 왜 다시 만들까요?
**앞선 RAG 단원**에서 만든 코드는 **`chromadb` 한 라이브러리의 사용법에 묶여** 있었습니다 — 컬렉션을 만들고 `add` 로 넣고 `query` 로 꺼내는 그 라이브러리만의 문법이죠. 벡터DB 를 바꾸는 순간 전부 다시 써야 합니다.

LangChain 은 이 단계들에 **공통 규약(부품 규약)** 을 정해 두었습니다. 규약을 지키면 **부품 하나만 갈아 끼워도 나머지 코드는 그대로**이고, 그 부품을 체인에 끼우거나 도구로 감쌀 수 있습니다 — 그게 오늘의 목적지입니다.

## 지난 단원과 나란히 놓고 보기

| 지난 단원(직접 호출) | 이번 단원(LangChain 부품) | 하는 일 |
|---|---|---|
| `pd.read_csv` 후 문자열 리스트 | `Document(page_content=, metadata=)` | 본문과 꼬리표를 함께 들고 다닌다 |
| 손으로 자른 청크 | `RecursiveCharacterTextSplitter` | 긴 글을 알맞은 크기로 자른다 |
| `SentenceTransformer(...).encode` | `HuggingFaceEmbeddings(model_name=...)` | 글을 벡터로 바꾼다 |
| `chromadb` 컬렉션 `add` / `query` | `Chroma.from_documents` / `as_retriever` | 벡터를 담고 가까운 것을 찾는다 |

**하는 일은 똑같습니다.** 이름과 규약만 바뀐 것입니다 — 그러니 지난 단원에서 이해한 원리를 그대로 들고 오세요.

<img src="images/rag_langchain_부품.png" width="820">

*문서를 Document 로 만들고 → 자르고 → 벡터로 바꿔 → 벡터스토어에 담으면 → 검색기가 된다.*

**(1단계) 표의 한 행을 `Document` 하나로 만듭니다.**

`Document` 는 **본문(`page_content`)** 과 **꼬리표(`metadata`)** 를 함께 들고 다니는 상자입니다. 검색은 `page_content` 로 하고, `metadata` 는 나중에 **출처를 표시**하거나 **범위를 좁힐 때** 씁니다. 지난 단원에서는 본문 리스트와 메타데이터 리스트를 따로 들고 다니며 순서를 맞춰야 했죠 — `Document` 는 둘을 **한 몸으로 묶어** 그 어긋남을 막아 줍니다.

In [ ]:
# 표 한 행 -> Document 하나. 사내 헬프데스크 FAQ 를 검색 대상으로 씁니다.
import pandas as pd
from langchain_core.documents import Document

hd_df = pd.read_csv('data/helpdesk_faq.csv')

# page_content 는 '검색 대상 본문', metadata 는 '함께 달고 다닐 꼬리표'입니다.
hd_docs = [Document(page_content=row.text, metadata={'id': row.id, 'title': row.title})
           for row in hd_df.itertuples()]

print('문서 수:', len(hd_docs))
print('본문 :', hd_docs[0].page_content)
print('꼬리표:', hd_docs[0].metadata)

**(2단계) 긴 글은 잘라야 합니다 — `RecursiveCharacterTextSplitter`.**

**자르는 규칙은 앞선 RAG 단원에서 손으로 만들어 봤습니다** — 조각이 너무 크면 여러 주제가 뭉개져 엉뚱한 글이 잡히고 너무 잘면 문장이 끊겨 뜻을 잃는다는 것도, 겹침을 왜 두는지도 거기서 했죠. **오늘 새로운 것은 하나뿐입니다 — 그 규칙이 이제 부품의 인자가 된다는 것.**

`RecursiveCharacterTextSplitter` 는 이름 그대로 **재귀적으로** 자릅니다 — 문단으로 나눠 보고, 그래도 크면 줄, 그래도 크면 문장·낱말로 내려갑니다. **큰 경계부터 존중**하는 것이죠. 인자는 둘입니다.

| 인자 | 뜻 |
|---|---|
| `chunk_size` | 한 조각의 최대 길이 |
| `chunk_overlap` | 앞 조각의 **끝을 얼마나 겹쳐** 다음 조각에 남길지 |

값을 고르는 법도 그때 얻은 결론 그대로입니다. 겹침을 올린다고 검색 품질이 **매끈하게 좋아지지 않았고**, 조각 크기가 달라지면 좋아지던 방향이 **뒤집히기까지** 했죠. 그러니 좋은 값은 감이 아니라 **재서** 고릅니다 — **재는 법은 그 단원에서 이미 익혔습니다**(Hit@K·Precision@K·Recall@K·MRR 로 겹침 값을 쓸어 재 봤지요). 여기서는 그때 얻은 결론대로 무난한 값 하나를 넣고 넘어갑니다.

In [ ]:
# 문서 목록을 조각 목록으로 자릅니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size=200 : 이 FAQ 한 편이 대략 100자 안팎이라 200자면 한 편이 통째로 들어갑니다.
# chunk_overlap=40 : 조각이 나뉠 때 앞 조각의 끝 40자를 겹쳐 둡니다.
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40)
hd_chunks = splitter.split_documents(hd_docs)

print('문서 수:', len(hd_docs), '-> 조각 수:', len(hd_chunks))
print('조각 길이:', [len(c.page_content) for c in hd_chunks])

# 확인 포인트: 조각도 Document 이고, 원래 문서의 metadata 가 그대로 따라붙었다는 것.
print('조각의 종류:', type(hd_chunks[0]).__name__)
print('조각의 꼬리표:', hd_chunks[0].metadata)

> **12개 문서가 12개 조각 — 하나도 잘리지 않았습니다.** 이 FAQ 는 한 편이 100자 안팎이라 `chunk_size=200` 안에 다 들어가기 때문입니다. 자르는 도구를 썼다고 늘 잘리는 것이 아니라 **크기를 넘는 글만** 잘립니다.

실제로 잘릴 때 무슨 일이 생기는지 **일부러 작은 크기**로 한 번 보겠습니다.

In [ ]:
# 일부러 작게 잘라, 실제로 쪼개질 때 무슨 일이 생기는지 봅니다(교육용 확인 — 실제 색인에는 쓰지 않습니다).
small_splitter = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=20)
small_chunks = small_splitter.split_documents(hd_docs[:1])   # 첫 문서 하나만

print('원문 길이:', len(hd_docs[0].page_content), '-> 조각', len(small_chunks), '개')
for i, c in enumerate(small_chunks):
    print(f'[{i}] 길이 {len(c.page_content)} · 꼬리표 {c.metadata}')
    print('   ', c.page_content)

> 두 가지를 확인하세요.

1. **꼬리표가 두 조각에 모두 복사되었습니다.** 하나의 문서에서 나온 조각들은 같은 `metadata` 를 갖습니다 — 그래서 나중에 조각이 검색되어도 **어느 문서에서 왔는지** 알 수 있습니다. 출처 표시가 가능한 이유입니다.
2. **앞 조각의 끝과 뒤 조각의 앞이 겹칩니다.** `chunk_overlap=20` 이 한 일입니다 — 앞선 RAG 단원에서 손으로 만들던 그 동작이 인자 하나로 바뀐 것뿐입니다.

**(3단계) 글을 벡터로 — `HuggingFaceEmbeddings`.**

**임베딩 단원과 RAG 단원**에서 쓰던 `SentenceTransformer` 그 모델을, 이번에는 **LangChain 임베딩 부품**으로 감싸 씁니다. 부품 규약은 두 가지입니다.

| 메서드 | 언제 쓰나 |
|---|---|
| `embed_documents(글 목록)` | 색인할 때 — 여러 글을 한꺼번에 벡터로 |
| `embed_query(글 하나)` | 검색할 때 — 질문 하나를 벡터로 |

이 규약만 지키면 **어떤 임베딩 모델이든 갈아 끼울 수 있습니다.** 우리는 한국어 문장을 다루므로 한국어로 학습된 모델을 씁니다.

In [ ]:
# 글을 벡터로 바꾸는 부품. 임베딩 단원에서 쓴 그 한국어 문장 임베딩 모델입니다.
from langchain_huggingface import HuggingFaceEmbeddings

# 처음 한 번은 모델을 내려받느라 시간이 걸립니다(그다음부터는 캐시에서 바로 읽습니다).
demo_embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')

query_vector = demo_embeddings.embed_query('VPN 접속')

# 확인 포인트: 짧은 질문 하나가 768개의 숫자로 바뀐다는 것 -- 임베딩 단원에서 본 그 차원입니다.
print('벡터 길이:', len(query_vector))
print('앞 5개  :', [round(v, 4) for v in query_vector[:5]])

**(4단계) 벡터를 담고, 검색기로 꺼냅니다 — `Chroma` 와 `as_retriever`.**

지난 단원에서는 클라이언트를 열고, 컬렉션을 만들고, 임베딩을 직접 계산해 `add` 하고, 질문 벡터를 만들어 `query` 했습니다. 네 단계였죠. 부품 규약에서는 **`Chroma.from_documents(문서들, 임베딩)`** 한 줄이 그 일을 전부 합니다 — 임베딩 계산도 안에서 알아서 합니다.

그리고 벡터스토어를 **`as_retriever()`** 로 감싸면 **검색기(Retriever)** 가 됩니다. 규약은 아주 단순합니다 — **문자열 질문을 넣으면 `Document` 목록이 나옵니다.** 그 단순함 덕분에 체인에 끼우고, 도구로 감싸고, **다른 종류의 검색기로 통째로 바꿔도** 쓰는 쪽 코드는 그대로입니다.

**인자 하나를 더 붙입니다 — `ids=`.** `ids` 를 주지 않으면 셀을 다시 실행할 때마다 새 id 가 붙어 **같은 글이 두 벌** 쌓이고, 검색 결과 두 자리를 같은 글이 차지합니다. 고유한 id 를 함께 넘기면 같은 id 는 **덮어쓰기**가 되어 몇 번을 실행해도 상태가 같습니다 — 컬렉션을 지웠다 다시 만들 것 없이 인자 하나로 끝납니다.

In [ ]:
# 조각들을 벡터로 만들어 저장소에 담습니다 -- 임베딩 계산은 from_documents 안에서 일어납니다.
from langchain_chroma import Chroma

# collection_name : 한 데이터베이스 안에서 이 색인을 구분하는 이름입니다.
# ids : 조각마다 고유한 이름을 붙여 넘깁니다 -> 같은 id 는 덮어쓰기라 여러 번 실행해도 중복되지 않습니다.
hd_store = Chroma.from_documents(hd_chunks, demo_embeddings, collection_name='helpdesk_demo',
                                 ids=[f'hd-{i}' for i in range(len(hd_chunks))])

# as_retriever : 저장소를 '검색기'로 감쌉니다.
#  search_kwargs={'k': 2} : 질문마다 가장 가까운 2개만 돌려줍니다.
#    k 가 크면 근거가 많아지지만 관련 없는 글까지 딸려 와 프롬프트가 길어집니다.
hd_retriever = hd_store.as_retriever(search_kwargs={'k': 2})

# 확인 포인트: 이 셀을 두 번, 세 번 실행해 보세요 -- 저장된 수가 늘지 않습니다(ids 덕분입니다).
print('저장된 조각 수:', len(hd_store.get()['ids']))

In [ ]:
# 검색기에 '문자열' 을 넣으면 'Document 목록' 이 나옵니다 -- 이것이 검색기의 규약입니다.
found = hd_retriever.invoke('재택에서 사내 시스템에 못 들어가요')

print('결과의 종류:', type(found).__name__, '/ 개수:', len(found))
print('원소의 종류:', type(found[0]).__name__)
for d in found:
    print('-', d.metadata['title'], '|', d.page_content[:40], '...')

> 질문에는 **VPN 이라는 낱말이 한 번도 나오지 않았는데** VPN 안내가 잡혔습니다. 낱말이 아니라 **뜻이 가까운** 것을 찾기 때문입니다 — 앞선 RAG 단원에서 본 그 성질 그대로입니다.

그리고 돌아온 것은 문자열이 아니라 **`Document` 목록**입니다. 본문뿐 아니라 `metadata` 가 함께 오므로 **답과 함께 출처를 붙일 수 있습니다.** 2절에서 바로 그 일을 합니다.

## 여기까지를 한 셀로 — 앞으로 쓸 색인

지금 손으로 만들어 본 네 단계를 한 셀에 모아 둡니다. **이 뒤의 모든 절은 여기서 만드는 `faq_retriever` 를 씁니다.** 앞의 시연에서 본 것처럼 이 FAQ 는 한 편이 짧아 자를 필요가 없으므로, 이 셀은 **문서를 그대로** 색인합니다(스플리터 단계 없이).

> **`ids=`** 는 앞 시연과 같은 이유로 붙어 있습니다. 다만 여기서는 이름을 새로 만들지 않고 **CSV 의 `id` 열을 그대로** 넘깁니다 — 이미 문서마다 고유한 값이 있으니까요.

In [ ]:
# [제공 코드] FAQ 문서를 LangChain 부품으로 색인합니다(임베딩 모델을 내려받느라 처음 한 번은 잠시 걸립니다).
import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

faq_df = pd.read_csv('data/helpdesk_faq.csv')

# 표의 한 행 = Document 하나. page_content 는 검색 대상 본문, metadata 는 함께 붙일 꼬리표입니다.
faq_docs = [Document(page_content=row.text, metadata={'id': row.id, 'title': row.title})
            for row in faq_df.itertuples()]

embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')   # 임베딩 단원에서 쓴 그 한국어 문장 임베딩 모델

# ids 를 함께 넘기면 같은 id 는 덮어쓰기가 됩니다 -> 이 셀을 여러 번 실행해도 문서가 중복되지 않습니다.
faq_store = Chroma.from_documents(faq_docs, embeddings, collection_name='helpdesk_kyoan',
                                  ids=faq_df['id'].tolist())
faq_retriever = faq_store.as_retriever(search_kwargs={'k': 2})   # 질문마다 가장 가까운 2개를 돌려주는 검색기

print('색인 완료 — 문서 수:', len(faq_docs))

### 🖐️ 함께 따라하기 — 서점 FAQ 를 같은 절차로 색인하기

이번에는 **온라인 서점 이용 안내 FAQ**(`data/bookstore_faq.csv`)로 같은 색인을 만들어 보세요. 컬럼 이름은 헬프데스크와 같습니다(`id`·`title`·`text`).

1. CSV 를 읽어 한 행을 `Document` 하나로 만든다(`page_content` 는 `text`, `metadata` 는 `id`·`title`).
2. **임베딩은 위에서 만든 `demo_embeddings` 를 그대로 쓴다** — 새로 만들면 모델을 또 올리게 됩니다.
3. `Chroma.from_documents(..., collection_name='bookstore_kyoan')` 로 담되, 위 셀처럼 **`ids=` 에 `id` 열을 넘겨** 여러 번 실행해도 중복되지 않게 한다. 그다음 `as_retriever(search_kwargs={'k': 2})` 로 **`book_retriever`** 를 만든다.
4. `'포인트는 얼마나 오래 쓸 수 있나요?'` 로 검색해 찾은 문서의 `title` 을 출력한다.

**확인 기준**: 포인트 관련 안내가 검색 결과에 들어 있다. (이 `book_retriever` 는 뒤 절의 따라하기에서 계속 씁니다.)

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) data/bookstore_faq.csv 를 pandas 로 읽는다
# 2) 한 행을 Document 하나로 만든다 (page_content=text, metadata 는 id 와 title)
# 3) Chroma.from_documents 로 담는다 — 임베딩은 위에서 만든 demo_embeddings 를 그대로 쓴다
#    (collection_name 은 'bookstore_kyoan', ids 에는 id 열을 넘겨 중복을 막는다)
# 4) as_retriever(search_kwargs={'k': 2}) 로 book_retriever 를 만든다
# 5) '포인트는 얼마나 오래 쓸 수 있나요?' 로 검색해 찾은 문서의 title 을 출력한다

### ✅ 바로 확인 퀴즈

**1.** `Document` 의 `page_content` 와 `metadata` 는 각각 무엇에 쓰이나요?

<details><summary>정답 보기</summary>

`page_content` 는 **검색 대상 본문**(이 글이 벡터가 됩니다). `metadata` 는 함께 달고 다니는 **꼬리표**로, 출처를 표시하거나 검색 범위를 좁힐 때 씁니다. 문서를 자르면 **꼬리표는 모든 조각에 복사**됩니다.

</details>

**2.** `chunk_overlap` 을 0 이 아니라 조금 주는 이유는 무엇인가요?

<details><summary>정답 보기</summary>

자르는 자리가 문장 한가운데일 수 있기 때문입니다. 겹침을 주면 경계에 걸린 문장이 **양쪽 조각에 모두** 남아, 어느 조각이 검색되어도 뜻이 통합니다. 다만 겹침이 크면 저장·검색 비용이 늘어납니다.

</details>

**3.** 검색기(`as_retriever(...)`)에 문자열을 넣으면 무엇이 나오나요?

<details><summary>정답 보기</summary>

**`Document` 목록**입니다. 문자열이 아니라 `Document` 라서 `metadata` 로 출처를 함께 쓸 수 있습니다.

</details>

---
# 2. RAG 체인 — LCEL 로 검색과 생성을 잇기

## 왜 체인으로 만들까요?
지금은 검색과 생성이 따로 놀고 있습니다. 검색해서 `Document` 목록을 받고, 그것을 사람이 프롬프트에 옮겨 담고, 모델을 부르고… 이 손작업을 **한 줄의 체인**으로 만들면 `invoke` 한 번에 끝납니다. 그리고 앞 단원에서 배운 `RunnablePassthrough`·`RunnableParallel` 이 바로 여기서 쓰입니다.

<img src="images/rag_체인_lcel.png" width="820">

*질문 하나가 두 갈래로 갈라져, 한쪽은 검색을 거쳐 자료가 되고 다른 한쪽은 그대로 질문으로 들어간다.*

**먼저 검색 결과를 프롬프트에 넣을 수 있는 모양으로 바꿉니다.** 검색기는 `Document` 목록을 주는데, 프롬프트의 `{context}` 자리에는 **글 한 덩어리**가 들어가야 합니다. 그 사이를 이어 주는 작은 함수를 만듭니다 — 제목을 함께 붙여 두면 모델이 어느 안내에서 나온 말인지 구분하기 좋습니다.

In [ ]:
# 검색 결과는 Document 목록입니다 — 프롬프트에 넣으려면 하나의 글로 합쳐야 합니다.
def format_docs(docs):
    """검색된 Document 들을 제목과 함께 한 덩어리 글로 합친다."""
    return '\n\n'.join(f"[{d.metadata['title']}] {d.page_content}" for d in docs)

In [ ]:
# 함수 하나만 따로 확인합니다 -- 체인에 끼우기 전에 부품 단위로 보는 습관이 디버깅에 좋습니다.
sample_docs = faq_retriever.invoke('비밀번호는 며칠마다 바꿔야 하나요?')
print(format_docs(sample_docs))

**(1단계) 프롬프트를 만듭니다.** system 에 **자료가 들어갈 자리**(`{context}`)를, human 에 **질문 자리**(`{question}`)를 둡니다.

여기서 한 문장이 결정적입니다 — **"주어진 자료에 있는 내용만으로 답하라"**. 이 말이 없으면 모델은 자료를 참고하되 **자기가 안다고 믿는 것을 섞어** 답하고, 사내 규정처럼 회사마다 다른 내용은 그 순간 **그럴듯하지만 틀린 답**이 됩니다. "자료에 없으면 없다고 답하라"까지 적어 두면 모델에게 **모른다고 말할 길**이 생깁니다 — RAG 로 환각을 줄이는 가장 기본적인 장치입니다.

In [ ]:
# 자료 자리({context})와 질문 자리({question})를 둔 프롬프트.
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    ('system',
     '너는 사내 헬프데스크 안내원이다. 아래 자료에 있는 내용만으로 한국어로 간결하게 답한다.\n'
     '자료에 없는 내용은 지어내지 말고 자료에서 찾지 못했다고 답한다.\n\n'
     '자료:\n{context}'),
    ('human', '{question}'),
])
print('프롬프트 변수:', rag_prompt.input_variables)   # context 와 question 두 자리

**(2단계) 체인을 조립합니다.** 핵심은 **맨 앞의 딕셔너리**입니다.

```python
{'context': faq_retriever | format_docs, 'question': RunnablePassthrough()}
```

체인에 딕셔너리를 두면 LangChain 이 그것을 `RunnableParallel` 로 바꿔 줍니다. 즉 **같은 입력(질문 문자열)이 두 갈래에 동시에** 들어갑니다.

- `context` 갈래: 질문 → 검색기 → `format_docs` → **자료 한 덩어리**
- `question` 갈래: 질문 → `RunnablePassthrough()` → **질문 그대로**

두 갈래의 결과가 `{'context': ..., 'question': ...}` 로 모이면, 그 모양이 바로 프롬프트가 원하는 모양입니다. 그래서 뒤에 `| rag_prompt | model | StrOutputParser()` 를 그대로 이을 수 있습니다.

In [ ]:
# 딕셔너리 안에서 두 갈래가 동시에 돌고, 그 결과가 프롬프트의 두 자리를 채웁니다.
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {'context': faq_retriever | format_docs,   # 질문 -> 검색 -> 한 덩어리 글
     'question': RunnablePassthrough()}        # 질문을 그대로 통과
    | rag_prompt
    | model
    | StrOutputParser()
)
print('RAG 체인 준비 완료')

**(3단계) 실행합니다.** 입력은 **문자열 하나**입니다 — 딕셔너리를 만들 필요가 없습니다. 두 자리를 채우는 일은 체인 안에서 일어나니까요.

In [ ]:
# 입력은 질문 문자열 하나. 검색·프롬프트 채우기·모델 호출·문자열 뽑기가 한 번에 일어납니다.
print(rag_chain.invoke('비밀번호는 며칠마다 바꿔야 하나요?'))

> 자료에 있는 내용으로 답이 나왔습니다. **모델이 원래 알고 있던 지식이 아니라** 우리가 넣어 준 사내 문서를 읽고 답한 것입니다 — 회사마다 다른 규정을 다룰 수 있는 이유가 이것입니다.

자료에 없는 것을 물어보면 어떻게 되는지도 한 번 시험해 보세요(예: 사내 식당 메뉴). 프롬프트에 적어 둔 규칙 덕분에 **지어내는 대신 못 찾았다고** 답할 것입니다.

**(4단계) 근거를 함께 돌려줍니다.**

지금 체인은 **답 문자열만** 줍니다. 실무에서 **출처 없는 RAG 답은 쓰기 어렵습니다** — "이게 정말 우리 회사 규정 맞아요?" 를 확인할 방법이 없고, 담당자도 **어느 문서를 고쳐야 할지** 모르며, 답이 이상할 때 **검색이 잘못된 것인지 생성이 잘못된 것인지** 가를 수도 없습니다.

**`RunnableParallel`** 을 쓰면 간단합니다 — 한쪽에는 방금 만든 체인을, 다른 한쪽에는 **검색기 자체**를 두면 됩니다. 같은 질문이 양쪽에 들어가므로 **답과 그 답의 근거가 짝을 이룹니다.**

In [ ]:
# 답과 근거를 한 번에 -- 같은 질문이 두 갈래(체인 / 검색기)에 동시에 들어갑니다.
from langchain_core.runnables import RunnableParallel

rag_with_sources = RunnableParallel(answer=rag_chain, sources=faq_retriever)

out = rag_with_sources.invoke('보안 USB 는 아무거나 써도 되나요?')
print('결과의 키:', list(out.keys()))
print('답  :', out['answer'])
print('출처:', [d.metadata['title'] for d in out['sources']])

> `sources` 에는 `Document` 목록이 그대로 들어 있으므로, 제목뿐 아니라 **문서 id·본문**까지 꺼내 화면에 "이 답변은 다음 문서를 참고했습니다" 로 붙일 수 있습니다. 1절에서 `metadata` 를 챙겨 둔 것이 여기서 값을 합니다.

### 🖐️ 함께 따라하기 — 서점 안내 RAG 체인

1절 따라하기에서 만든 `book_retriever` 로 **서점 안내용 RAG 체인**을 만들어 보세요.

1. system 에 `{context}`, human 에 `{question}` 을 둔 프롬프트를 만든다(역할은 "온라인 서점 고객센터 안내원", **자료에 있는 내용만으로 답하라**는 규칙을 꼭 넣는다).
2. `{'context': book_retriever | format_docs, 'question': RunnablePassthrough()} | 프롬프트 | model | StrOutputParser()` 로 `book_chain` 을 만든다.
3. `'전자책은 몇 대까지 볼 수 있어요?'` 를 `invoke` 해 답을 출력한다.

**확인 기준**: `invoke` 에 **문자열 하나**만 넣었는데 자료를 근거로 한 답이 나온다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) system 에 {context}, human 에 {question} 을 둔 서점 안내용 프롬프트를 만든다
#    (자료에 있는 내용만으로 답하라는 규칙을 반드시 넣는다)
# 2) {'context': book_retriever | format_docs, 'question': RunnablePassthrough()}
#    | 프롬프트 | model | StrOutputParser() 로 book_chain 을 만든다
# 3) '전자책은 몇 대까지 볼 수 있어요?' 를 invoke 해 답을 출력한다

### ✅ 바로 확인 퀴즈

**1.** 체인 맨 앞의 `{'context': ..., 'question': RunnablePassthrough()}` 는 무슨 일을 하나요?

<details><summary>정답 보기</summary>

**같은 입력(질문 문자열)을 두 갈래로 동시에 흘려** 보냅니다. `context` 갈래는 검색을 거쳐 자료 한 덩어리가 되고, `question` 갈래는 `RunnablePassthrough` 로 **질문 그대로** 통과합니다. 두 결과가 딕셔너리로 모여 프롬프트의 두 자리를 채웁니다.

</details>

**2.** 프롬프트에 "자료에 있는 내용만으로 답하라"를 넣는 이유는?

<details><summary>정답 보기</summary>

이 말이 없으면 모델이 **자기가 알고 있다고 믿는 일반 지식을 섞어** 답합니다. 사내 규정처럼 조직마다 다른 내용은 그 순간 그럴듯하지만 틀린 답이 됩니다. "없으면 없다고 답하라"까지 적어 두면 모델이 모른다고 말할 길이 생깁니다.

</details>

**3.** 답과 함께 출처를 돌려주려면 어떻게 하나요?

<details><summary>정답 보기</summary>

`RunnableParallel(answer=체인, sources=검색기)` 로 묶습니다. 같은 질문이 양쪽에 들어가므로 답과 근거가 짝을 이루고, `sources` 의 `Document` 에서 `metadata` 를 꺼내 출처로 보여 줄 수 있습니다.

</details>

## 체인의 한계 — 인사말에도 검색이 돈다

방금 만든 체인은 훌륭하지만 한 가지 성질이 있습니다. **무엇을 묻든 반드시 검색합니다.** 파이프로 고정된 경로이기 때문입니다. 인사말을 넣어 보면 바로 드러납니다.

In [ ]:
# 인사말을 검색기에 넣어 봅니다 -- 검색은 '가장 가까운 2개' 를 무조건 돌려줍니다.
greet_docs = faq_retriever.invoke('고마워요, 좋은 하루 보내세요!')
print('인사말로 검색한 결과:', [d.metadata['title'] for d in greet_docs])

In [ ]:
# 같은 인사말을 체인에 넣으면, 위의 엉뚱한 자료가 그대로 프롬프트에 붙어 모델에게 갑니다.
print(rag_chain.invoke('고마워요, 좋은 하루 보내세요!'))

> 인사말과 아무 상관 없는 안내문이 잡혔고, 답도 인사에 어울리지 않습니다. **벡터 검색은 '관련 없음' 이라고 답하지 않습니다** — 언제나 *상대적으로* 가장 가까운 `k` 개를 돌려줍니다. 체인은 그 결과를 무조건 프롬프트에 붙입니다.

**이 문제는 여기서 풀지 않습니다.** 검색을 *할지 말지* 를 판단하게 하려면 경로를 고정하지 않는 다른 구조가 필요한데, 그것은 **다음 시간**의 주제입니다. 오늘은 대신 검색으로는 아예 답할 수 없는 질문 쪽으로 갑니다 — **"보안 USB 재고가 몇 개야?"** 같은 것입니다.

---
# 3. Text-to-SQL — 자연어 질문을 SQL 로

## 아이디어
회사의 데이터는 문서에만 있는 것이 아닙니다. **재고·주문·매출은 데이터베이스의 표**에 있습니다. 이런 질문은 검색으로 답할 수 없습니다 — 세어 보고 더해 봐야 알 수 있으니까요. 그것을 하는 언어가 **SQL·데이터베이스 단원**에서 배운 **SQL** 입니다.

그래서 이렇게 합니다.

1. **표 구조를 프롬프트로 알려 준다** (어떤 표에 어떤 열이 있는지)
2. **모델이 SQL 을 만든다**
3. **우리가 그 SQL 을 실행한다**

여기서 잊으면 안 되는 사실이 하나 있습니다. **모델은 데이터베이스를 볼 수 없습니다.** 표가 몇 개인지, 열 이름이 무엇인지, 어느 열이 어느 표를 가리키는지 — 전부 **우리가 프롬프트로 적어 준 것만** 압니다. **스키마 설명이 곧 모델의 눈입니다.** 설명이 부실하면 모델은 존재하지 않는 열 이름을 지어냅니다.

먼저 데이터베이스를 준비합니다. **SQL·데이터베이스 단원**에서 배운 **sqlite** 를 그대로 씁니다 — 접속 정보가 필요 없어 바로 시작할 수 있습니다. 연결을 **두 개** 만드는 것에 주목하세요. 하나는 우리가 눈으로 볼 때 쓰는 보통 연결이고, 다른 하나는 **에이전트에게 줄 읽기 전용 연결**입니다. 왜 그렇게 하는지는 곧 나옵니다.

In [ ]:
# [제공 코드] 데이터베이스 준비 — SQL 단원에서 배운 sqlite 를 그대로 씁니다(접속 정보가 필요 없습니다).
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('output') / 'helpdesk.db'
DB_PATH.parent.mkdir(exist_ok=True)
DB_PATH.unlink(missing_ok=True)          # 여러 번 실행해도 늘 같은 초기 상태에서 시작합니다

_conn = sqlite3.connect(DB_PATH, isolation_level=None)   # isolation_level=None : 실행 즉시 저장
_conn.execute('pragma foreign_keys = on')                # 외래키 검사를 켭니다(기본값은 꺼짐)
_conn.executescript(Path('data/setup_day19.sql').read_text(encoding='utf-8'))
_conn.execute('pragma foreign_keys = on')                # executescript 뒤에 한 번 더 켭니다

# 에이전트에게 줄 연결은 따로 만들고 '읽기 전용'으로 엽니다 — 모델이 무슨 SQL 을 만들든 쓰기가 막힙니다.
#  check_same_thread=False : 에이전트는 도구를 별도 스레드에서 실행하므로 이 옵션이 없으면 도구가 전부 실패합니다.
_ro_conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True,
                           isolation_level=None, check_same_thread=False)


def run_query(sql):
    """SELECT 결과를 DataFrame 으로 돌려준다(사람이 눈으로 확인할 때 쓴다)."""
    cur = _conn.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])


print('데이터베이스 준비 완료 —', DB_PATH)

## 데이터 살펴보기

질문을 만들기 전에 **어떤 데이터가 있는지 먼저 봅니다.** 사내 비품 목록과 부서별 주문 내역, 두 표입니다.

In [ ]:
# 비품 목록 -- 이름·분류·단가·재고.
display(run_query('select * from hd_item'))

In [ ]:
# 부서별 주문 내역 -- item_id 로 위의 비품 표와 이어집니다(SQL 단원에서 배운 외래키).
display(run_query('select * from hd_order'))

## ⚠️ 그 전에 — 위험을 먼저 봅니다

**모델이 만든 SQL 을 그대로 실행한다**는 말을 다시 읽어 보세요. 이것은 **모르는 사람이 써 준 코드를 내 데이터베이스에서 실행**하는 것과 같습니다. 모델이 악의를 갖는다는 뜻이 아닙니다 — 질문을 잘못 알아듣고 `delete` 를 만들 수도 있고, 사용자가 프롬프트에 교묘한 지시를 섞어 넣을 수도 있습니다.

그래서 가드를 **두 겹**으로 겁니다.

| 겹 | 무엇을 하나 | 무엇을 막나 |
|---|---|---|
| 1) 문자열 검사 | `select` 로 시작하는 **한 문장**만 통과 | 대놓고 쓰는 `delete`·여러 문장 |
| 2) 읽기 전용 연결 | 연결 자체가 쓰기를 거부 | **1겹을 빠져나간 모든 쓰기** |

<img src="images/text_to_sql_두겹가드.png" width="820">

*질문에서 SQL 이 만들어지면 문자열 검사를 지나고, 읽기 전용 연결에서만 실행된다.*

In [ ]:
# [제공 코드] 데이터베이스 조회 도구 — 에이전트가 이 도구로 SQL 을 실행합니다.
#  가드가 두 겹입니다: (1) 여기서 문장을 검사하고 (2) 연결 자체가 읽기 전용입니다.
from langchain_core.tools import tool


@tool
def run_select(sql: str) -> str:
    """읽기 전용 SQL(SELECT) 한 문장을 실행하고 결과를 문자열로 돌려준다. SELECT 한 문장이 아니면 거부한다."""
    stmt = sql.strip().rstrip(';')          # 끝의 세미콜론 하나는 흔한 표기라 허용한다
    # 세미콜론이 남아 있으면 문장이 둘 이상이라는 뜻 — 'select 1; delete ...' 를 막는다.
    if not stmt.lower().startswith('select') or ';' in stmt:
        return '거부: 이 도구는 SELECT 조회 한 문장만 실행할 수 있습니다.'
    try:
        return str(_ro_conn.execute(stmt).fetchall())   # 검사한 문장을 그대로 실행한다
    except Exception as e:
        return f'에러: {e}'                             # 에러도 문자열로 — 모델이 읽고 고쳐 다시 시도한다


print('SQL 도구 준비:', run_select.name)

**첫 번째 겹을 시험합니다.** 네 가지 SQL 을 넣어 무엇이 통과하고 무엇이 막히는지 보세요. **네 번째가 이 절의 핵심**입니다 — 미리 결과를 예상해 보세요.

In [ ]:
# 문자열 검사가 무엇을 막고 무엇을 통과시키는지 직접 확인합니다.
attempts = [
    'select count(*) from hd_order',                      # 평범한 조회
    'select 1; delete from hd_order',                     # 조회인 척하며 뒤에 삭제를 붙임
    'delete from hd_order',                               # 대놓고 삭제
    'select 1 from hd_order where 1=0 union select 1',    # 조회이지만 복잡한 문장
]
for sql in attempts:
    print(sql)
    print('   ->', run_select.invoke({'sql': sql}))

> 앞의 세 개는 예상대로입니다. 문제는 **네 번째**입니다.

`select 1 from hd_order where 1=0 union select 1` 은 **문자열 검사를 통과합니다.** `select` 로 시작하고 세미콜론도 없으니 검사 기준으로는 흠잡을 데가 없기 때문입니다. 실제로는 두 조회를 `union` 으로 붙인 문장인데도 말이죠.

**교훈은 이렇습니다.** 문자열 검사는 "SELECT 인가" 만 봅니다. 그러니 아무리 정교한 조회라도 SELECT 이기만 하면 다 통과합니다. 검사식을 더 촘촘하게 만들어 볼 수도 있겠지만, **금지 목록을 늘리는 방식은 언제나 빠져나갈 구멍이 남습니다.** 그래서 두 번째 겹이 필요합니다.

**두 번째 겹을 시험합니다.** 도구를 거치지 않고 **읽기 전용 연결에 직접** 삭제를 시도해 보겠습니다. 문자열 검사를 아예 건너뛴 셈이니, 막을 것이 남아 있다면 그것이 진짜 방어선입니다.

In [ ]:
# 문자열 검사를 건너뛰고 읽기 전용 연결에 직접 삭제를 시도합니다.
#   try 로 감싼 이유: 여기서 에러가 나는 것이 '정상' 이고, 그 에러 문구를 보여 주려는 것입니다.
try:
    _ro_conn.execute('delete from hd_order')
    print('삭제되었습니다 -- 이러면 안 됩니다')
except Exception as e:
    print('막혔습니다:', e)

# 확인 포인트: 데이터가 그대로 남아 있는지 -- 보통 연결로 다시 세어 봅니다.
print('남은 주문 건수:', run_query('select count(*) as cnt from hd_order').loc[0, 'cnt'])

> **`attempt to write a readonly database`** — 연결 자체가 쓰기를 거부했습니다. 이것이 **진짜 방어선**입니다.

차이를 정리하면 이렇습니다. 문자열 검사는 **막을 것을 하나씩 골라내는** 방식이라 새로운 수법이 나오면 뚫립니다. 읽기 전용 연결은 **할 수 있는 일 자체를 좁혀 두는** 방식이라, 어떤 SQL 이 오든 쓰기는 불가능합니다. 실무에서 데이터베이스 권한을 조회 전용 계정으로 분리하는 것이 바로 이 발상입니다.

그렇다고 첫 번째 겹이 쓸모없는 것은 아닙니다. **에러가 나기 전에** 거부하면 로그가 깔끔하고, 모델에게 "이건 안 된다"는 메시지를 돌려줘 **다시 시도하게** 할 수 있습니다.

## 스키마 프롬프트 — 모델의 눈을 만들어 주기

이제 모델에게 표 구조를 알려 줍니다. 좋은 스키마 프롬프트에는 네 가지가 들어갑니다.

1. **표와 열 이름·자료형** — 없는 열을 지어내지 않도록
2. **표끼리의 관계** — `hd_order.item_id` 가 `hd_item.item_id` 를 가리킨다는 사실(JOIN 을 할 수 있으려면 필수)
3. **각 표가 무엇을 담는지** 한 줄 설명 — 이름만으로는 뜻이 안 보이므로
4. **지켜야 할 규칙** — SELECT 한 문장만, 반드시 도구로 실행해 확인한 값으로 답할 것

In [ ]:
# 모델은 데이터베이스를 볼 수 없습니다 -- 이 글이 모델이 아는 전부입니다.
schema_prompt = """너는 사내 비품 데이터베이스를 조회해 답하는 도우미다.
아래 표만 존재한다. 반드시 run_select 도구로 SQL 을 실행해 확인한 값으로 답한다.

표 구조(sqlite):
  hd_item(item_id text, item_name text, category text, unit_price int, stock int)
    -- 비품 목록. unit_price 는 단가(원), stock 은 현재 재고 수량.
  hd_order(order_id int, item_id text, quantity int, dept text, order_date text)
    -- 부서별 주문 내역. dept 는 주문한 부서 이름.
  hd_order.item_id 는 hd_item.item_id 를 가리킨다.

규칙:
- SELECT 한 문장만 만든다.
- 결과를 사람이 읽을 한국어 문장으로 정리해 답한다."""

print(schema_prompt)

In [ ]:
# 스키마 설명을 system_prompt 로 주고, 조회 도구 하나를 붙인 에이전트.
from langchain.agents import create_agent

sql_agent = create_agent(model, [run_select], system_prompt=schema_prompt)


def called_tools(res):
    """에이전트 궤적에서 (도구 이름, 인자) 목록을 뽑는다 -- 무엇을 불렀는지 확인용."""
    return [(c['name'], c['args']) for m in res['messages']
            if getattr(m, 'tool_calls', None) for c in m.tool_calls]


def show_sql(res):
    """에이전트가 만든 SQL 과 최종 답을 함께 보여 준다."""
    # 궤적에는 모델이 도구를 부른 기록이 남아 있습니다 -- 거기서 sql 인자를 꺼냅니다.
    for _, args in called_tools(res):
        print('만든 SQL:', args['sql'])
    print('답      :', res['messages'][-1].text)


print('Text-to-SQL 에이전트 준비 완료')

In [ ]:
# (1) 조건을 걸어 세는 질문 -- WHERE 와 COUNT 가 필요합니다.
show_sql(sql_agent.invoke({'messages': '개발팀이 주문한 건수는 몇 건이야?'}))

In [ ]:
# (2) 극값을 찾는 질문 -- 정렬(ORDER BY)과 LIMIT 이 필요합니다.
show_sql(sql_agent.invoke({'messages': '가장 비싼 비품은 뭐야?'}))

In [ ]:
# (3) 묶어서 세는 질문 -- GROUP BY 가 필요합니다.
show_sql(sql_agent.invoke({'messages': '부서별 주문 건수를 많은 순으로 알려줘.'}))

In [ ]:
# (4) 두 표를 이어야 답이 나오는 질문 -- 금액은 주문 수량(hd_order)과 단가(hd_item)를 곱해야 합니다.
show_sql(sql_agent.invoke({'messages': '주문 금액이 가장 큰 부서는 어디야?'}))

> 네 질문의 SQL 을 나란히 보세요. `WHERE`·`COUNT` 에서 시작해 `ORDER BY`·`LIMIT`, `GROUP BY` 를 지나 마지막에는 **두 표를 `JOIN` 하고 `SUM(quantity * unit_price)` 까지** 만들었습니다.

**우리가 준 것은 표 구조 설명 한 덩어리뿐입니다** — 어느 열을 곱하라고도, 어느 표를 이으라고도 하지 않았습니다. 모델은 `hd_order.item_id 는 hd_item.item_id 를 가리킨다` 는 한 줄에서 JOIN 조건을 세우고, "주문 금액" 이라는 말에서 수량과 단가의 곱을 떠올린 것입니다. **스키마 설명의 품질이 곧 답의 품질**인 이유가 이것입니다.

(실호출이라 만들어지는 SQL 의 표현은 실행할 때마다 조금씩 다를 수 있습니다. `as` 별칭이나 대소문자가 달라도 뜻이 같으면 같은 답이 나옵니다.)

## 구조화된 출력으로 SQL 받기

지금은 모델이 도구를 부르는 김에 SQL 을 만들었습니다. **SQL 만 따로, 정해진 모양으로** 받고 싶다면 지난 시간에 쓴 그 **`with_structured_output`** 을 그대로 씌우면 됩니다. 그러면 SQL 만이 아니라 **근거와 사용한 표까지** 함께 옵니다.

- **감사 로그**: 어떤 질문이 어떤 SQL 이 되었고 왜 그렇게 판단했는지 그대로 남길 수 있습니다.
- **사전 점검**: 실행하기 전에 `tables` 를 보고 **허용된 표만 건드리는지** 코드로 확인할 수 있습니다.
- **사람의 승인**: 위험한 조회는 실행 전에 담당자에게 SQL 과 이유를 보여 주고 승인을 받을 수 있습니다.

요약하면 — **에이전트에 맡기면 실행까지 알아서** 되고, **구조화된 출력으로 받으면 실행 직전에 우리가 끼어들 수 있습니다.**

In [ ]:
# 받고 싶은 모양을 스키마로 선언합니다. Field 의 description 이 모델에게 가는 설명입니다.
from pydantic import BaseModel, Field


class SqlPlan(BaseModel):
    """자연어 질문을 조회 계획으로 옮긴 것."""

    sql: str = Field(description='실행할 SELECT 한 문장')
    reason: str = Field(description='이 SQL 로 질문에 답이 되는 이유 한 문장')
    tables: list[str] = Field(description='이 SQL 이 사용하는 표 이름 목록')


print('스키마 필드:', list(SqlPlan.model_fields))

In [ ]:
# 모델에 스키마를 씌우고 '스키마 설명 + 질문' 을 문자열 하나로 넣습니다.
plan = model.with_structured_output(SqlPlan).invoke(schema_prompt + '\n\n질문: 주변기기 분류 비품의 평균 단가는?')

print('결과의 종류:', type(plan).__name__)   # SqlPlan -- 문자열이 아니라 객체입니다
print('sql   :', plan.sql)
print('reason:', plan.reason)
print('tables:', plan.tables)

In [ ]:
# 받은 계획을 '점검한 뒤' 실행합니다 -- 실행 직전에 우리가 끼어들 수 있다는 것이 핵심입니다.
allowed_tables = {'hd_item', 'hd_order'}

if set(plan.tables) <= allowed_tables:          # 허용한 표만 쓰는지 확인
    print('실행 결과:', run_select.invoke({'sql': plan.sql}))
else:
    print('실행하지 않음 -- 허용되지 않은 표:', set(plan.tables) - allowed_tables)

> `plan` 은 문자열이 아니라 **객체**입니다 — `plan.sql`·`plan.reason`·`plan.tables` 를 따로 꺼내 로그 한 줄로 남길 수 있습니다. "이 답은 왜 이렇게 나왔지?" 를 되짚을 때 그 기록이 있고 없고는 큰 차이입니다.

### 🖐️ 함께 따라하기 — 서점 데이터베이스에 Text-to-SQL

같은 데이터베이스에 **온라인 서점 표 세 개**가 함께 들어 있습니다. 이 표들을 조회하는 에이전트를 만들어 보세요.

| 표 | 열 |
|---|---|
| `bs_customer` | `customer_id`·`name`·`grade`·`city` |
| `bs_book` | `book_id`·`title`·`author`·`genre`·`price`·`stock` |
| `bs_order` | `order_id`·`customer_id`·`book_id`·`quantity`·`order_date` |

1. 위 세 표를 설명하는 `bs_prompt` 를 만든다. **표끼리의 관계**(`bs_order.customer_id` → `bs_customer.customer_id`, `bs_order.book_id` → `bs_book.book_id`)를 반드시 적는다.
2. `create_agent(model, [run_select], system_prompt=bs_prompt)` 로 `book_sql_agent` 를 만든다.
3. `'장르별 재고 합계를 많은 순으로 알려줘.'` 를 `invoke` 하고 `show_sql` 로 만들어진 SQL 과 답을 확인한다.

**확인 기준**: `GROUP BY` 가 들어간 SQL 이 만들어지고 장르별 합계가 나온다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) bs_customer·bs_book·bs_order 세 표를 설명하는 bs_prompt 를 만든다
#    (열 이름·자료형, 표끼리의 관계, SELECT 한 문장만 만들라는 규칙을 포함)
# 2) create_agent(model, [run_select], system_prompt=bs_prompt) 로 book_sql_agent 를 만든다
# 3) '장르별 재고 합계를 많은 순으로 알려줘.' 를 invoke 하고 show_sql 로 SQL 과 답을 확인한다

### ✅ 바로 확인 퀴즈

**1.** 모델은 데이터베이스를 볼 수 있나요? 없다면 무엇을 보고 SQL 을 만드나요?

<details><summary>정답 보기</summary>

볼 수 없습니다. **우리가 프롬프트에 적어 준 스키마 설명이 모델이 아는 전부**입니다. 그래서 열 이름·자료형과 **표끼리의 관계**를 정확히 적어 줘야 하고, 설명이 부실하면 없는 열을 지어냅니다.

</details>

**2.** `select 1 from hd_order where 1=0 union select 1` 은 왜 문자열 검사를 통과하나요? 그래서 무엇이 진짜 방어선인가요?

<details><summary>정답 보기</summary>

`select` 로 시작하고 세미콜론이 없어 검사 기준을 만족하기 때문입니다. 문자열 검사는 "SELECT 인가" 만 보므로 정교한 조회는 다 통과합니다. **진짜 방어선은 읽기 전용 연결**입니다 — 어떤 SQL 이 오든 쓰기 자체가 불가능합니다(`attempt to write a readonly database`).

</details>

**3.** SQL 을 `with_structured_output` 으로 받으면 무엇이 좋은가요?

<details><summary>정답 보기</summary>

SQL 만이 아니라 **근거(`reason`)와 사용한 표(`tables`)까지 함께** 받습니다. 그래서 감사 로그로 남기고, 실행 전에 허용된 표만 쓰는지 코드로 점검하고, 필요하면 사람의 승인을 받을 수 있습니다.

</details>

---
## 이번 강의 정리

| 개념 | 핵심 | 코드 |
|---|---|---|
| Document | 본문과 꼬리표를 한 몸으로 | `Document(page_content=, metadata=)` |
| 스플리터 | 큰 글만 자르고, 겹침으로 경계를 보호 | `RecursiveCharacterTextSplitter(chunk_size=, chunk_overlap=)` |
| 임베딩 부품 | 어떤 임베딩이든 같은 규약으로 | `HuggingFaceEmbeddings(model_name=)` |
| 벡터스토어·검색기 | 문자열을 넣으면 `Document` 목록 | `Chroma.from_documents(...).as_retriever(...)` |
| RAG 체인 | 검색 갈래와 질문 갈래를 동시에 | `{'context': 검색기 \| format_docs, 'question': RunnablePassthrough()}` |
| 근거 함께 받기 | 출처 없는 답은 실무에서 못 쓴다 | `RunnableParallel(answer=..., sources=검색기)` |
| Text-to-SQL | 스키마 설명이 모델의 눈 | `create_agent(model, [run_select], system_prompt=schema_prompt)` |
| 두 겹 가드 | 문자열 검사 + **읽기 전용 연결** | `mode=ro` 연결이 진짜 방어선 |
| 구조화된 SQL | 근거·표까지 받아 로그로 | `model.with_structured_output(SqlPlan)` |

- **부품 규약**을 쓰면 데이터가 바뀌어도(헬프데스크 → 서점) 코드 모양이 그대로입니다.
- **체인은 무엇을 묻든 검색합니다** — 인사말에도 검색이 돕니다. 이 문제는 다음 시간에 해결합니다.
- 모델이 만든 SQL 을 실행할 때 **막을 것을 고르는 검사보다, 할 수 있는 일을 좁히는 권한**이 강합니다.

## ⏭️ 예고 — 다음 시간

오늘 우리는 검색과 조회를 **만들었습니다.** 하지만 하나가 남았습니다 — 2절 끝에서 본 그 문제, **체인은 인사말에도 검색이 돈다**는 것입니다.

다음 시간에는 경로를 우리가 고정하지 않고 **에이전트가 스스로 고르게** 합니다. 오늘 만든 검색과 조회가 그대로 그 에이전트의 **도구**가 됩니다. 그리고 도구가 여러 개가 되는 순간 새 문제가 생깁니다 — **모델이 무엇을 보고 도구를 고르는가**, 그리고 **잘못 고를 때 어떻게 고치는가**. 그리고 **RAG 단원에서 익힌 그 지표**로, 이번에는 에이전트가 쓰는 검색을 다시 재 봅니다.

수고하셨습니다!